In [36]:
import pandas as pd
from sqlalchemy import create_engine, text
from typing import Union
import warnings
warnings.filterwarnings('ignore')
from DATA.stock_invest_function import *

def fetch_valuation_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = 'US_company_valuation_result') -> Union[pd.DataFrame, None]:


    """
    지정한 ticker의 밸류에이션 결과를 조회하여 DataFrame으로 반환.
    결과가 없으면 메시지 출력 후 None 반환.
    """
    # 안전을 위해 입력 정규화 (대문자화/공백제거)
    q_ticker = (ticker or "").strip().upper()
    if not q_ticker:
        print("ticker 값을 올바르게 입력해 주세요.")
        return None

    # SQLAlchemy 엔진 생성
    connection_string = (
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    engine = create_engine(connection_string)

    try:
        # 테이블 존재 여부(옵션): 없으면 깔끔히 리턴
        with engine.connect() as conn:
            # MariaDB/ MySQL에서 information_schema로 존재 체크
            exists_sql = text("""
                SELECT COUNT(*) AS cnt
                FROM information_schema.tables
                WHERE table_schema = :schema AND table_name = :table
            """)
            exists = conn.execute(exists_sql, {"schema": db_info['database'], "table": table_name}).scalar()
            if not exists:
                print(f"테이블 '{table_name}' 이(가) 존재하지 않습니다.")
                return None

            # 본 조회 (ticker 컬럼 기준; 필요한 경우 인덱스 사용 권장)
            query = text(f"""
                SELECT *
                FROM {table_name}
                WHERE UPPER(ticker) = :ticker
                ORDER BY 1
            """)
            df = pd.read_sql(query, conn, params={"ticker": q_ticker})

        if df.empty:
            print("조회한  ticker의 기업 valation이 존재하지 않습니다")
            return None

        # datetime 컬럼 자동 파싱이 필요하면 여기서 처리 가능
        # for col in df.columns:
        #     if col.lower().endswith(('date', 'dt', 'time', 'timestamp')):
        #         df[col] = pd.to_datetime(df[col], errors='ignore')

        return df

    except Exception as e:
        print(f"조회 중 오류가 발생했습니다: {e}")
        return None
    finally:
        engine.dispose()

# MariaDB 연결 및 고유 ticker 추출
def get_unique_tickers():
   engine = create_engine(f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}")

   query = "SELECT DISTINCT ticker FROM US_company_valuation_result"
   unique_tickers = pd.read_sql(query, engine)

   return unique_tickers['ticker'].tolist()


def get_financial_data(tickers, api_key):
   """
   Financial Modeling Prep API를 통해 기업별 재무 데이터 수집

   Parameters:
   tickers: list - 티커 리스트
   api_key: str - API 키
   """

   results = []

   for ticker in tickers:
       try:
           # 1. 최근 4분기 매출 데이터 (Income Statement)
           income_url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}?period=quarter&limit=4&apikey={api_key}"
           income_response = requests.get(income_url)
           income_data = income_response.json()

           # 최근 4분기 매출 합산
           revenue_4q = sum([quarter.get('revenue', 0) for quarter in income_data]) if income_data else 0

           # 2. ROA 데이터 (Financial Ratios)
           ratios_url = f"https://financialmodelingprep.com/api/v3/ratios-ttm/{ticker}?apikey={api_key}"
           ratios_response = requests.get(ratios_url)
           ratios_data = ratios_response.json()

           roa = ratios_data[0].get('returnOnAssetsTTM', None) if ratios_data else None

           # 3. 시가총액 데이터 (Market Cap)
           market_cap_url = f"https://financialmodelingprep.com/api/v3/market-capitalization/{ticker}?apikey={api_key}"
           market_cap_response = requests.get(market_cap_url)
           market_cap_data = market_cap_response.json()

           market_cap = market_cap_data[0].get('marketCap', None) if market_cap_data else None

           # 결과 저장
           results.append({
               'ticker': ticker,
               'revenue_4q_sum': revenue_4q,
               'roa': roa,
               'market_cap': market_cap
           })

           print(f"{ticker} 데이터 수집 완료")

       except Exception as e:
           print(f"{ticker} 데이터 수집 실패: {e}")
           results.append({
               'ticker': ticker,
               'revenue_4q_sum': None,
               'roa': None,
               'market_cap': None
           })

   # DataFrame 생성
   df = pd.DataFrame(results)
   return df


In [46]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

df = fetch_valuation_by_ticker(db_info, "TTD")
#     print(df.head())   # 앞부분 출력
print("=== 데이터 기본 정보 ===")
print(f"데이터 형태: {df.shape}")
print(f"컬럼: {list(df.columns)}")

# 각 indicator별 데이터 개수 확인
print("\n=== 각 indicator별 데이터 개수 ===")
indicator_counts = df['indicator'].value_counts()
print(indicator_counts)

# 1. revenue_with_exog, revenue_without_exog 데이터 합산
# forward_revenue, revenue 모두 포함
revenue_with_exog_sum = (
    df[df['indicator'] == 'forward_revenue_with_exog']['value'].sum() +
    df[df['indicator'] == 'revenue_with_exog']['value'].sum()
)

revenue_without_exog_sum = (
    df[df['indicator'] == 'forward_revenue_without_exog']['value'].sum() +
    df[df['indicator'] == 'revenue_without_exog']['value'].sum()
)

revenue_with_exog_sum = (
    df[df['indicator'] == 'revenue_with_exog']['value'].sum()
)

revenue_without_exog_sum = (
    df[df['indicator'] == 'revenue_without_exog']['value'].sum()
)

print("\n=== 매출 합산 결과 ===")
print(f"향후 4분기 예상매출_외생반영: {revenue_with_exog_sum:.2f}")
print(f"향후 4분기 예상매출_외생미반영: {revenue_without_exog_sum:.2f}")

# 2. Value 관련 데이터 추출 및 평균 계산

# value_with_exog - 실제 컬럼명 사용, 값에 4를 곱한 후 평균 계산
value_with_exog_data = df[df['indicator'] == 'value_with_exog']['value'] * 4
value_with_exog_avg = value_with_exog_data.mean() if not value_with_exog_data.empty else 0

# value_without_exog - 실제 컬럼명 사용, 값에 4를 곱한 후 평균 계산
value_without_exog_data = df[df['indicator'] == 'value_without_exog']['value'] * 4
value_without_exog_avg = value_without_exog_data.mean() if not value_without_exog_data.empty else 0

# lstm_value_with_exog - 실제 컬럼명 사용
lstm_with_exog_data = df[df['indicator'] == 'lstm_value_with_exog']['value']
lstm_with_exog_avg = lstm_with_exog_data.mean() if not lstm_with_exog_data.empty else 0

# lstm_value_without_exog - 실제 컬럼명 사용
lstm_without_exog_data = df[df['indicator'] == 'lstm_value_without_exog']['value']
lstm_without_exog_avg = lstm_without_exog_data.mean() if not lstm_without_exog_data.empty else 0

# prophet_value_with_exog - 실제 컬럼명 사용
prophet_with_exog_data = df[df['indicator'] == 'prophet_value_with_exog']['value']
prophet_with_exog_avg = prophet_with_exog_data.mean() if not prophet_with_exog_data.empty else 0

# prophet_value_without_exog - 실제 컬럼명 사용
prophet_without_exog_data = df[df['indicator'] == 'prophet_value_without_exog']['value']
prophet_without_exog_avg = prophet_without_exog_data.mean() if not prophet_without_exog_data.empty else 0

# value_monthly_sarima_with_exog - 실제 컬럼명 사용
monthly_sarima_with_exog_data = df[df['indicator'] == 'value_monthly_sarima_with_exog']['value']
monthly_sarima_with_exog_avg = monthly_sarima_with_exog_data.mean() if not monthly_sarima_with_exog_data.empty else 0

# value_monthly_sarima_without_exog - 실제 컬럼명 사용
monthly_sarima_without_exog_data = df[df['indicator'] == 'value_monthly_sarima_without_exog']['value']
monthly_sarima_without_exog_avg = monthly_sarima_without_exog_data.mean() if not monthly_sarima_without_exog_data.empty else 0

print("\n=== 각 모델별 평균값 ===")
print(f"value_with_exog_평균: {value_with_exog_avg:.6f}")
print(f"value_without_exog_평균: {value_without_exog_avg:.6f}")
print(f"lstm_value_with_exog_평균: {lstm_with_exog_avg:.6f}")
print(f"lstm_value_without_exog_평균: {lstm_without_exog_avg:.6f}")
print(f"prophet_value_with_exog_평균: {prophet_with_exog_avg:.6f}")
print(f"prophet_value_without_exog_평균: {prophet_without_exog_avg:.6f}")
print(f"value_monthly_sarima_with_exog_평균: {monthly_sarima_with_exog_avg:.6f}")
print(f"value_monthly_sarima_without_exog_평균: {monthly_sarima_without_exog_avg:.6f}")

# 각 지표별 데이터 개수도 출력
print("\n=== 각 지표별 데이터 개수 확인 ===")
indicators_to_check = [
    'value_with_exog', 'value_without_exog',
    'lstm_value_with_exog', 'lstm_value_without_exog',
    'prophet_value_with_exog', 'prophet_value_without_exog',
    'value_monthly_sarima_with_exog', 'value_monthly_sarima_without_exog'
]

for indicator in indicators_to_check:
    count = len(df[df['indicator'] == indicator])
    avg_value = df[df['indicator'] == indicator]['value'].mean()
    print(f"{indicator}: {count}개 데이터, 평균값: {avg_value:.2f}")

# 3. mean_market_value 계산 (value_with_exog_평균, value_without_exog_평균 제외)
mean_market_value_list = [
    lstm_with_exog_avg,
    lstm_without_exog_avg,
    prophet_with_exog_avg,
    prophet_without_exog_avg,
    monthly_sarima_with_exog_avg,
    monthly_sarima_without_exog_avg
]

# 0보다 큰 값들만 포함하여 평균 계산
valid_values = [v for v in mean_market_value_list if v > 0]
mean_market_value = np.mean(valid_values) if valid_values else 0

print(f"\n=== 최종 mean_market_value ===")
print(f"mean_market_value 계산에 포함된 지표 개수: {len(valid_values)}")
print(f"mean_market_value: {mean_market_value:.6f}")

# 4. 결과를 딕셔너리로 정리
results = {
    '향후 4분기 예상매출_외생반영': revenue_with_exog_sum,
    '향후 4분기 예상매출_외생미반영': revenue_without_exog_sum,
    'value_with_exog_평균': value_with_exog_avg,
    'value_without_exog_평균': value_without_exog_avg,
    'lstm_value_with_exog_평균': lstm_with_exog_avg,
    'lstm_value_without_exog_평균': lstm_without_exog_avg,
    'prophet_value_with_exog_평균': prophet_with_exog_avg,
    'prophet_value_without_exog_평균': prophet_without_exog_avg,
    'value_monthly_sarima_with_exog_평균': monthly_sarima_with_exog_avg,
    'value_monthly_sarima_without_exog_평균': monthly_sarima_without_exog_avg,
    'mean_market_value': mean_market_value
}

# 5. 결과를 DataFrame으로 변환
result_df = pd.DataFrame([results])

print("\n=== 최종 결과 DataFrame ===")
result_transposed = result_df.T
result_transposed.columns = ['값']
print(result_transposed)

# 6. CSV로 저장 (선택사항)
# result_df.to_csv('forecast_results.csv', index=False, encoding='utf-8-sig')
# print("\n결과가 'forecast_results.csv' 파일로 저장되었습니다.")

=== 데이터 기본 정보 ===
데이터 형태: (100, 9)
컬럼: ['frequency', 'ticker', 'forecast_date', 'indicator', 'value', 'exog_var', 'params', 'target_date', 'valuation_time']

=== 각 indicator별 데이터 개수 ===
indicator
forward_revenue_without_exog         16
PSR_monthly_sarima_without_exog      12
value_monthly_sarima_without_exog    12
forecasted_PSR_lstm                  12
forecasted_PSR_prophet               12
lstm_value_without_exog              12
prophet_value_without_exog           12
PSR_quarter_without_exog              4
revenue_without_exog                  4
value_without_exog                    4
Name: count, dtype: int64

=== 매출 합산 결과 ===
향후 4분기 예상매출_외생반영: 0.00
향후 4분기 예상매출_외생미반영: 3122.75

=== 각 모델별 평균값 ===
value_with_exog_평균: 0.000000
value_without_exog_평균: 429072.069536
lstm_value_with_exog_평균: 0.000000
lstm_value_without_exog_평균: 49333819050.666664
prophet_value_with_exog_평균: 0.000000
prophet_value_without_exog_평균: 37959614746.031883
value_monthly_sarima_with_exog_평균: 0.000000
value_monthly

In [44]:
# 실행
tickers = get_unique_tickers()
print(f"고유 ticker 개수: {len(tickers)}")
print(f"ticker 목록: {tickers}")

고유 ticker 개수: 45
ticker 목록: ['AVGO', 'META', 'ABBV', 'BLKB', 'CARR', 'CCK', 'CHRW', 'CSL', 'EMR', 'FI', 'GE', 'HRL', 'KMB', 'MNST', 'NVR', 'PLXS', 'SAM', 'SLVM', 'TPR', 'VRTX', 'VVV', 'DDOG', 'NX', 'ANET', 'UI', 'ORCL', 'DXCM', 'DRRX', 'MELI', 'TTD', 'LOVE', 'INVX', 'WDAY', 'PANW', 'GMED', 'CRM', 'CRUS', 'CAT', 'CSCO', 'EXTR', 'AEIS', 'HON', 'AMD', 'SMCI', 'APH']


In [33]:
def create_forecast_dataframe(db_info):
   engine = create_engine(f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}")

   # 전체 데이터 조회
   query = "SELECT * FROM US_company_valuation_result"
   df = pd.read_sql(query, engine)

   # 1. 고유 ticker 추출
   unique_tickers = df['ticker'].unique()

   # 결과 DataFrame 초기화
   result_df = pd.DataFrame({'ticker': unique_tickers})

   # mean_market_value 계산용 indicator 목록
   market_value_indicators = [
       'prophet_value_without_exog',
       'prophet_value_with_exog',
       'lstm_value_without_exog',
       'lstm_value_with_exog',
       'value_monthly_sarima_without_exog',
       'value_monthly_sarima_with_exog'
   ]

   # ticker별로 데이터 처리
   for ticker in unique_tickers:
       ticker_data = df[df['ticker'] == ticker]

       # 2. exog_var 값 추출
       exog_var = ticker_data['exog_var'].iloc[0] if len(ticker_data) > 0 else None

       # 3. forecast_sale 계산
       revenue_with_exog = ticker_data[ticker_data['indicator'] == 'revenue_with_exog']['value'].sum()
       revenue_without_exog = ticker_data[ticker_data['indicator'] == 'revenue_without_exog']['value'].sum()
       forecast_sale = (revenue_with_exog + revenue_without_exog) / 2

       # 4. mean_market_value 계산
       market_values = []
       for indicator in market_value_indicators:
           values = ticker_data[ticker_data['indicator'] == indicator]['value']
           # 양수인 값들만 평균 계산에 포함
           positive_values = values[values > 0]
           if len(positive_values) > 0:
               market_values.extend(positive_values.tolist())

       mean_market_value = sum(market_values) / len(market_values) if market_values else None

       # 결과에 추가
       idx = result_df[result_df['ticker'] == ticker].index[0]
       result_df.loc[idx, 'exog_var'] = exog_var
       result_df.loc[idx, 'forecast_sale'] = forecast_sale
       result_df.loc[idx, 'mean_market_value'] = mean_market_value

   # revenue_exog_sum 컬럼 추가 (빈 값으로)
   result_df['revenue_exog_sum'] = None

   return result_df



In [34]:
# 실행
forecast_df = create_forecast_dataframe(db_info)
print(forecast_df)

   ticker exog_var  forecast_sale  mean_market_value revenue_exog_sum
0    AVGO     None   99013.197943       1.824253e+12             None
1    META   854239  199369.810273       9.260595e+11             None
2    ABBV   300290   62466.989710       4.331654e+11             None
3    BLKB   851762    1139.399414       3.600466e+09             None
4    CARR   854141   22892.767373       6.223207e+10             None
5     CCK   760429   12441.089013       1.084568e+10             None
6    CHRW   392321   17121.729042       1.974533e+10             None
7     CSL   300213    5177.508290       1.883504e+10             None
8     EMR   842139   18691.174311       7.497518e+10             None
9      FI     None   11064.441717       8.133257e+10             None
10     GE   841191   41801.129718       3.062621e+11             None
11    HRL     None    6177.955153       1.347591e+10             None
12    KMB   440131   17822.353132       4.458042e+10             None
13   MNST     None  

In [37]:
apikey = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
financial_df = get_financial_data(tickers, apikey)
print(financial_df)

AVGO 데이터 수집 완료
META 데이터 수집 완료
ABBV 데이터 수집 완료
BLKB 데이터 수집 완료
CARR 데이터 수집 완료
CCK 데이터 수집 완료
CHRW 데이터 수집 완료
CSL 데이터 수집 완료
EMR 데이터 수집 완료
FI 데이터 수집 완료
GE 데이터 수집 완료
HRL 데이터 수집 완료
KMB 데이터 수집 완료
MNST 데이터 수집 완료
NVR 데이터 수집 완료
PLXS 데이터 수집 완료
SAM 데이터 수집 완료
SLVM 데이터 수집 완료
TPR 데이터 수집 완료
VRTX 데이터 수집 완료
VVV 데이터 수집 완료
DDOG 데이터 수집 완료
NX 데이터 수집 완료
ANET 데이터 수집 완료
UI 데이터 수집 완료
ORCL 데이터 수집 완료
DXCM 데이터 수집 완료
DRRX 데이터 수집 완료
MELI 데이터 수집 완료
TTD 데이터 수집 완료
LOVE 데이터 수집 완료
INVX 데이터 수집 완료
WDAY 데이터 수집 완료
PANW 데이터 수집 완료
GMED 데이터 수집 완료
CRM 데이터 수집 완료
CRUS 데이터 수집 완료
CAT 데이터 수집 완료
CSCO 데이터 수집 완료
EXTR 데이터 수집 완료
AEIS 데이터 수집 완료
HON 데이터 수집 완료
AMD 데이터 수집 완료
SMCI 데이터 수집 완료
APH 데이터 수집 완료
   ticker  revenue_4q_sum         roa     market_cap
0    AVGO     57046000000    0.078461  1436439738000
1    META    178804000000    0.242607  1893399878054
2    ABBV     58328000000    0.027680   375870971200
3    BLKB      1141002000   -0.106251     3199634236
4    CARR     22463000000    0.104383    53852735440
5     CCK     12013000000    0

In [41]:
pd.merge(forecast_df, financial_df, on='ticker', how='left').to_excel(r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\포트폴리오_후보+리스트_2025_3q_port.xlsx')